# Binary Heart Disease Classification 

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve
)
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

In [ ]:
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
SEEDS = [7, 21, 42]
print('Seeds:', SEEDS)

## Data Source and Provenance



In [ ]:
column_names = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'diagnosis'
]

candidate_paths = [
    Path('/home/rohanprashant/Downloads/heart+disease/processed.cleveland.data'),
    Path('/home/rohanprashant/Downloads/heart+disease/cleveland.data')
]

df = None
used_source = None
for p in candidate_paths:
    if p.exists():
        df = pd.read_csv(p, header=None, names=column_names)
        used_source = str(p)
        break

if df is None:
    backup_url = 'https://raw.githubusercontent.com/dataprofessor/data/master/heart-disease-cleveland.csv'
    df = pd.read_csv(backup_url)
    df.columns = [c.strip() for c in df.columns]
    used_source = backup_url

df = df.replace('?', np.nan)
for c in column_names:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

print('Data source used:', used_source)
print('Shape:', df.shape)
df.head()

## Data Quality Audit



In [ ]:
raw_df = df.copy()

print('Duplicate rows:', raw_df.duplicated().sum())
print('Missing values by column:')
display(raw_df.isna().sum().to_frame('missing_count').T)

clean_df = raw_df.dropna().copy()
clean_df['diagnosis_binary'] = (clean_df['diagnosis'] > 0).astype(int)

print('Rows after dropping missing values:', len(clean_df))
print('Binary class distribution (0=absent, 1=present):')
display(clean_df['diagnosis_binary'].value_counts(normalize=True).rename('proportion').to_frame())

num_cols = [c for c in clean_df.columns if c not in ['diagnosis', 'diagnosis_binary']]
outlier_summary = {}
for col in num_cols:
    q1 = clean_df[col].quantile(0.25)
    q3 = clean_df[col].quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    outlier_summary[col] = int(((clean_df[col] < low) | (clean_df[col] > high)).sum())

display(pd.Series(outlier_summary, name='iqr_outlier_count').sort_values(ascending=False).head(10).to_frame())

## Problem Framing


In [ ]:
feature_cols = [c for c in clean_df.columns if c not in ['diagnosis', 'diagnosis_binary']]
X = clean_df[feature_cols].values
y = clean_df['diagnosis_binary'].values

# Stratified split: 70/15/15
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print('Train size:', len(y_train), 'Val size:', len(y_val), 'Test size:', len(y_test))
print('Train positive rate:', y_train.mean().round(3))
print('Val positive rate  :', y_val.mean().round(3))
print('Test positive rate :', y_test.mean().round(3))

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
X_val_t = torch.tensor(X_val_s, dtype=torch.float32)
X_test_t = torch.tensor(X_test_s, dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.long)
y_val_t = torch.tensor(y_val, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

## Neural Network (Logits + CrossEntropyLoss)



In [ ]:
class BinaryNet(nn.Module):
    def __init__(self, input_dim, hidden_size, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 2)
        )

    def forward(self, x):
        return self.net(x)

def train_model(model, X_tr, y_tr, X_v, y_v, lr, weight_decay, epochs=150, batch_size=32):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)

    train_losses, val_losses = [], []
    for _ in range(epochs):
        model.train()
        epoch_loss = 0.0
        for xb, yb in loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(xb)

        train_losses.append(epoch_loss / len(X_tr))

        model.eval()
        with torch.no_grad():
            val_logits = model(X_v)
            val_loss = criterion(val_logits, y_v).item()
        val_losses.append(val_loss)

    return train_losses, val_losses

def predict_proba(model, X):
    model.eval()
    with torch.no_grad():
        logits = model(X)
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
    return probs

In [ ]:
def threshold_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
        'sensitivity': sensitivity,
        'specificity': specificity,
        'confusion_matrix': np.array([[tn, fp], [fn, tp]]),
        'threshold': threshold
    }

def find_threshold_for_sensitivity(y_true, y_prob, target_sensitivity=0.85):
    thresholds = np.linspace(0.05, 0.95, 181)
    best_t, best_gap = 0.5, float('inf')
    for t in thresholds:
        m = threshold_metrics(y_true, y_prob, threshold=t)
        gap = abs(m['sensitivity'] - target_sensitivity)
        if gap < best_gap:
            best_gap, best_t = gap, t
    return best_t

## Hyperparameter Search with Multi-Seed Robustness



In [ ]:
param_grid = {
    'lr': [1e-2, 1e-3],
    'hidden_size': [32, 64, 128],
    'dropout': [0.0, 0.3],
    'weight_decay': [0.0, 1e-4],
}

input_dim = X_train_t.shape[1]
search_results = []

for cfg in ParameterGrid(param_grid):
    seed_scores = []
    for seed in SEEDS:
        set_seed(seed)
        model = BinaryNet(input_dim, cfg['hidden_size'], cfg['dropout'])
        train_model(
            model, X_train_t, y_train_t, X_val_t, y_val_t,
            lr=cfg['lr'], weight_decay=cfg['weight_decay'], epochs=120
        )
        val_prob = predict_proba(model, X_val_t)
        seed_scores.append(roc_auc_score(y_val, val_prob))

    row = dict(cfg)
    row['val_auc_mean'] = float(np.mean(seed_scores))
    row['val_auc_std'] = float(np.std(seed_scores))
    search_results.append(row)

search_df = pd.DataFrame(search_results).sort_values('val_auc_mean', ascending=False).reset_index(drop=True)
display(search_df.head(8))
best_cfg = search_df.iloc[0].to_dict()
best_cfg

In [ ]:
models = []
seed_level_metrics = []

for seed in SEEDS:
    set_seed(seed)
    model = BinaryNet(input_dim, int(best_cfg['hidden_size']), float(best_cfg['dropout']))
    train_losses, val_losses = train_model(
        model, X_train_t, y_train_t, X_val_t, y_val_t,
        lr=float(best_cfg['lr']),
        weight_decay=float(best_cfg['weight_decay']),
        epochs=150
    )

    val_prob = predict_proba(model, X_val_t)
    test_prob = predict_proba(model, X_test_t)
    thr = find_threshold_for_sensitivity(y_val, val_prob, target_sensitivity=0.85)

    m = threshold_metrics(y_test, test_prob, threshold=thr)
    m['roc_auc'] = roc_auc_score(y_test, test_prob)
    m['pr_auc'] = average_precision_score(y_test, test_prob)
    m['seed'] = seed
    m['val_threshold'] = thr

    seed_level_metrics.append(m)
    models.append({'seed': seed, 'model': model, 'train_losses': train_losses, 'val_losses': val_losses, 'test_prob': test_prob})

metrics_df = pd.DataFrame(seed_level_metrics)[['seed', 'accuracy', 'f1', 'sensitivity', 'specificity', 'roc_auc', 'pr_auc', 'val_threshold']]
display(metrics_df)
print('Mean +/- std across seeds')
display(metrics_df.drop(columns=['seed', 'val_threshold']).agg(['mean', 'std']))

best_seed_row = metrics_df.sort_values('roc_auc', ascending=False).iloc[0]
best_seed = int(best_seed_row['seed'])
best_model_entry = [m for m in models if m['seed'] == best_seed][0]
best_test_prob = best_model_entry['test_prob']
best_threshold = float(best_seed_row['val_threshold'])
print('Selected representative seed:', best_seed)
print('Representative threshold:', round(best_threshold, 3))

In [ ]:
best_metrics = threshold_metrics(y_test, best_test_prob, threshold=best_threshold)
best_metrics['roc_auc'] = roc_auc_score(y_test, best_test_prob)
best_metrics['pr_auc'] = average_precision_score(y_test, best_test_prob)

print('Binary test metrics (representative best-seed model)')
for k in ['accuracy', 'f1', 'sensitivity', 'specificity', 'roc_auc', 'pr_auc', 'threshold']:
    print(f'{k:12s}: {best_metrics[k]:.4f}')

cm = best_metrics['confusion_matrix']
plt.figure(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred 0', 'Pred 1'], yticklabels=['True 0', 'True 1'])
plt.title('Confusion Matrix (Test)')
plt.tight_layout()
plt.show()

fpr, tpr, _ = roc_curve(y_test, best_test_prob)
prec, rec, _ = precision_recall_curve(y_test, best_test_prob)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(fpr, tpr, label=f'AUC={best_metrics["roc_auc"]:.3f}')
ax[0].plot([0, 1], [0, 1], linestyle='--', color='gray')
ax[0].set_title('ROC Curve')
ax[0].set_xlabel('False Positive Rate')
ax[0].set_ylabel('True Positive Rate')
ax[0].legend()

ax[1].plot(rec, prec, label=f'PR-AUC={best_metrics["pr_auc"]:.3f}')
ax[1].set_title('Precision-Recall Curve')
ax[1].set_xlabel('Recall')
ax[1].set_ylabel('Precision')
ax[1].legend()

plt.tight_layout()
plt.show()

## Baseline Comparison: Logistic Regression



In [ ]:
lr_model = LogisticRegression(max_iter=2000, random_state=42)
lr_model.fit(X_train_s, y_train)

lr_val_prob = lr_model.predict_proba(X_val_s)[:, 1]
lr_test_prob = lr_model.predict_proba(X_test_s)[:, 1]
lr_thr = find_threshold_for_sensitivity(y_val, lr_val_prob, target_sensitivity=0.85)
lr_metrics = threshold_metrics(y_test, lr_test_prob, threshold=lr_thr)
lr_metrics['roc_auc'] = roc_auc_score(y_test, lr_test_prob)
lr_metrics['pr_auc'] = average_precision_score(y_test, lr_test_prob)

comparison = pd.DataFrame([
    {
        'model': 'NeuralNet',
        'accuracy': best_metrics['accuracy'],
        'f1': best_metrics['f1'],
        'sensitivity': best_metrics['sensitivity'],
        'specificity': best_metrics['specificity'],
        'roc_auc': best_metrics['roc_auc'],
        'pr_auc': best_metrics['pr_auc']
    },
    {
        'model': 'LogisticRegression',
        'accuracy': lr_metrics['accuracy'],
        'f1': lr_metrics['f1'],
        'sensitivity': lr_metrics['sensitivity'],
        'specificity': lr_metrics['specificity'],
        'roc_auc': lr_metrics['roc_auc'],
        'pr_auc': lr_metrics['pr_auc']
    }
]).set_index('model')
display(comparison.round(4))

## Interpretability



In [ ]:
perm = permutation_importance(
    lr_model, X_test_s, y_test,
    scoring='f1', n_repeats=20, random_state=42
)

perm_df = pd.DataFrame({
    'feature': feature_cols,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
}).sort_values('importance_mean', ascending=False)

display(perm_df.head(10))

plt.figure(figsize=(8, 5))
sns.barplot(data=perm_df.head(10), x='importance_mean', y='feature', color='steelblue')
plt.title('Top 10 Permutation Importances (Logistic Baseline)')
plt.xlabel('Mean Importance (F1 drop)')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

print('Optional SHAP output (if shap package is available):')
try:
    import shap
    explainer = shap.LinearExplainer(lr_model, X_train_s)
    shap_values = explainer.shap_values(X_test_s)
    shap.summary_plot(shap_values, X_test_s, feature_names=feature_cols, show=True)
except Exception as e:
    print('SHAP not available or failed to run:', e)

## Clinical Interpretation Note



In [ ]:
top_features = perm_df.head(5)['feature'].tolist()
print('Top 5 model-influential features:', top_features)

for f in top_features:
    corr = np.corrcoef(clean_df[f].values, clean_df['diagnosis_binary'].values)[0, 1]
    print(f'{f:10s} correlation with binary diagnosis: {corr:.3f}')

## Fairness and Subgroup Monitoring



In [ ]:
test_df = pd.DataFrame(X_test, columns=feature_cols)
test_df['y_true'] = y_test
test_df['y_prob'] = best_test_prob
test_df['y_pred'] = (test_df['y_prob'] >= best_threshold).astype(int)

def subgroup_metrics(df_sub):
    if len(df_sub) < 5 or df_sub['y_true'].nunique() < 2:
        return {'n': len(df_sub), 'f1': np.nan, 'sensitivity': np.nan, 'specificity': np.nan}
    m = threshold_metrics(df_sub['y_true'].values, df_sub['y_prob'].values, threshold=best_threshold)
    return {'n': len(df_sub), 'f1': m['f1'], 'sensitivity': m['sensitivity'], 'specificity': m['specificity']}

# Sex subgroup (assuming 0=female, 1=male in this dataset variant)
sex_rows = []
for sex_value, g in test_df.groupby('sex'):
    r = subgroup_metrics(g)
    r['group'] = f'sex={int(sex_value)}'
    sex_rows.append(r)

# Age subgroup
test_df['age_group'] = np.where(test_df['age'] >= 60, 'age>=60', 'age<60')
age_rows = []
for age_label, g in test_df.groupby('age_group'):
    r = subgroup_metrics(g)
    r['group'] = age_label
    age_rows.append(r)

subgroup_df = pd.DataFrame(sex_rows + age_rows)[['group', 'n', 'f1', 'sensitivity', 'specificity']]
display(subgroup_df.round(4))

## Model Card Style Summary

### Intended Use
Decision-support risk triage for heart disease screening, not a standalone diagnostic system.

### Data
Tabular clinical features from Cleveland-style heart disease dataset. Missing rows removed in this prototype.

### Performance
Report metrics above with seed mean and standard deviation. Prioritize sensitivity for safety-critical triage workflows.

### Key Risks
- Bias across age and sex groups
- Dataset shift across hospitals/populations
- Small sample size and overfitting risk
- Missing/noisy clinical inputs

### Mitigations
- Subgroup monitoring dashboard
- Periodic retraining and calibration checks
- Human-in-the-loop clinician review for low-confidence cases
- Threshold tuning according to clinical objective and false-negative budget

## Actionable Recommendations for IntelliSys

### Proposed Inference Pipeline
1. Intake and schema validation
2. Missing-data checks and preprocessing
3. Model inference and confidence scoring
4. Clinician decision support screen
5. Logging and monitoring (quality, drift, subgroup metrics)

### Operational KPIs
- Sensitivity target (for example >= 0.85)
- False alarm rate ceiling
- Alert volume per day
- Subgroup performance gap thresholds

### Rollout Plan
- Phase 1: Offline validation and governance sign-off
- Phase 2: Shadow mode in clinical workflow
- Phase 3: Limited pilot with audit checkpoints
- Phase 4: Controlled scale-up with monthly model review

## Extension Notes for Other Group Notebooks

To align all group notebooks with this standard:
- In multiclass notebook, report macro-F1, weighted-F1, per-class recall, and multiclass confusion matrix.
- In dual-head notebook, report metrics separately per head and include joint error analysis.
- Keep the same model card and risk/mitigation framing for consistency in final presentation.